# MAL Model Experiments

Use this notebook to compare candidate models without putting experimental training code in the FastAPI import path. Production training and inference helpers live in `app/model.py`; this notebook imports those helpers and only keeps exploratory orchestration here.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

repo_or_mal_dir = Path.cwd()
mal_dir = repo_or_mal_dir if (repo_or_mal_dir / "app").exists() else repo_or_mal_dir / "MAL"
sys.path.insert(0, str(mal_dir))

from app.model import (  # noqa: E402
    DATASET_PATH,
    FEATURE_COLUMNS,
    MODEL_PATH,
    RANDOM_STATE,
    REAL_DATASET_PATH,
    TARGET_COLUMN,
    evaluate_model,
    load_dataset,
    load_real_sensor_dataset,
    save_model,
    train_validation_test_split,
)

In [2]:
df = load_dataset(DATASET_PATH)

display(df.head())
print(f"Rows: {len(df):,}")
print(f"Features: {FEATURE_COLUMNS}")
display(df[TARGET_COLUMN].value_counts().sort_index().rename("rows_per_rating"))

,timePeriod,currentTemperature,currentHumidity,currentCO2,currentLight,currentNoise,maxTemp,minTemp,meanTemp,maxHumidity,...,maxCO2,minCO2,meanCO2,maxLight,minLight,meanLight,maxNoise,minNoise,meanNoise,rating
0,2026-04-01 08:49:00 to 2026-04-01 09:19:00,22,58,815,325,56,23,22,22.05,58,...,848,631,723.55,325,306,316.00,62,43,51.20,4
1,2026-04-01 10:39:00 to 2026-04-01 11:09:00,26,47,751,799,60,26,26,26.00,48,...,765,699,728.80,801,782,790.10,69,53,59.70,3
2,2026-04-02 19:15:00 to 2026-04-02 19:45:00,24,61,1230,630,56,27,24,25.39,61,...,1230,702,906.32,637,617,627.35,56,36,47.32,2
3,2026-04-05 02:57:00 to 2026-04-05 03:27:00,19,52,825,597,35,20,19,19.25,53,...,932,691,794.36,614,595,605.57,54,34,43.75,3
4,2026-04-05 07:32:00 to 2026-04-05 08:02:00,24,54,1534,658,34,27,23,25.00,71,...,1534,782,1099.48,674,654,662.35,48,30,37.39,1


Rows: 1,811
Features: ['currentTemperature', 'maxTemp', 'minTemp', 'meanTemp']


rating
1    271
2    360
3    474
4    393
5    313
Name: rows_per_rating, dtype: int64

In [3]:
if REAL_DATASET_PATH.exists():
    real_sensor_df = load_real_sensor_dataset(REAL_DATASET_PATH)
    display(real_sensor_df.head())
    print(f"Collected real sensor rows: {len(real_sensor_df):,}")
else:
    print(f"No collected real sensor dataset yet: {REAL_DATASET_PATH}")

No collected real sensor dataset yet: /home/edf/Documents/GitSynced/SEP4_/MAL/app/environment_history_realdata.csv


In [4]:
x_train, x_validation, x_test, y_train, y_validation, y_test = train_validation_test_split(df)

print(f"Train rows: {len(x_train):,}")
print(f"Validation rows: {len(x_validation):,}")
print(f"Test rows: {len(x_test):,}")

Train rows: 1,158
Validation rows: 290
Test rows: 363


In [5]:
candidate_models = {
    "decision_tree": DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=3),
    "random_forest": RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100),
    "gradient_boosting": GradientBoostingClassifier(random_state=RANDOM_STATE, n_estimators=100),
}

results = []
trained_models = {}

for name, model in candidate_models.items():
    model.fit(x_train, y_train)
    trained_models[name] = model
    validation_metrics = evaluate_model(model, x_validation, y_validation)
    test_metrics = evaluate_model(model, x_test, y_test)
    results.append(
        {
            "model": name,
            "validation_accuracy": validation_metrics["accuracy"],
            "validation_macro_f1": validation_metrics["macro_f1"],
            "test_accuracy": test_metrics["accuracy"],
            "test_macro_f1": test_metrics["macro_f1"],
        }
    )

metrics_df = pd.DataFrame(results).sort_values("validation_macro_f1", ascending=False)
display(metrics_df)

,model,validation_accuracy,validation_macro_f1,test_accuracy,test_macro_f1
1,random_forest,0.279310,0.277204,0.234160,0.220229
2,gradient_boosting,0.265517,0.266647,0.234160,0.221436
0,decision_tree,0.268966,0.155523,0.300275,0.166331


In [6]:
best_model_name = metrics_df.iloc[0]["model"]
best_model = trained_models[best_model_name]

print(f"Best validation model: {best_model_name}")

if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=FEATURE_COLUMNS)
    display(importances.sort_values(ascending=False).rename("importance"))

Best validation model: random_forest


meanTemp              0.643132
currentTemperature    0.141114
maxTemp               0.114089
minTemp               0.101664
Name: importance, dtype: float64